# Part A evidence log — review of the inherited route classifier

The previous engineer's `baseline/baseline_classifier.py` reports **98.75% accuracy** and
recommends shipping. This notebook is the evidence log behind my review: every claim I make in
the README is *measured* here, on the real data, before I assume it. Structure:

1. The dataset at a glance
2. Reverse-engineering the generator
3. Reproducing 98.75% and testing the leakage claim
4. What an 80-row test set can certify
5. Two protocols, two questions — and the gap between them
6. Capacity: 1,522 features for 400 rows
7. Adversarial probe: false positives from the learned features
8. The other direction: does it catch *novel* fraud?
9. Regularization as an operating point
10. Calibration and the human-review band
11. Alternatives (incl. Naive Bayes), with significance testing
12. Hygiene candidate: `min_df=2`
13. Embeddings — considered, rejected
14. Verdict

In [1]:
import csv, re, warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")
pd.set_option("display.width", 120)

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB, ComplementNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, StratifiedKFold, GroupKFold, cross_val_predict
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix,
                             f1_score, recall_score)

SEED = 0
DATA = Path("../data/train.csv") if Path("../data/train.csv").exists() else Path("data/train.csv")
rows = list(csv.DictReader(open(DATA, encoding="utf-8")))
texts = np.array([r["text"] for r in rows], dtype=object)
labels = np.array([r["label"] for r in rows])
ROUTES = sorted(set(labels))

def baseline_config(**kw):
    """The previous engineer's exact model config, as a leakage-proof pipeline."""
    return make_pipeline(
        TfidfVectorizer(ngram_range=(1, 2), min_df=kw.get("min_df", 1), sublinear_tf=True),
        LogisticRegression(max_iter=2000, C=10.0, class_weight=kw.get("class_weight")),
    )

import sklearn
print(f"{len(rows)} rows | sklearn {sklearn.__version__} | numpy {np.__version__}")

400 rows | sklearn 1.8.0 | numpy 1.26.4


## 1. The dataset at a glance

Four routes with a 3.2:1 spread between the most and least common class. `fraud-report` — the
route the brief says is *most expensive to get wrong* — is the rarest at 12.5%. Any evaluation
that blends the classes (accuracy) is structurally biased against noticing fraud mistakes.

In [2]:
counts = pd.Series(labels).value_counts()
summary = pd.DataFrame({
    "count": counts,
    "share": (counts / len(labels)).round(3),
    "balanced_class_weight n/(K*n_c)": (len(labels) / (len(ROUTES) * counts)).round(3),
})
lengths = pd.Series([len(t.split()) for t in texts])
print(summary, "\n")
print(f"message length (words): min {lengths.min()}, median {lengths.median():.0f}, max {lengths.max()}")

                     count  share  balanced_class_weight n/(K*n_c)
general                160  0.400                            0.625
account-access         100  0.250                            1.000
transaction-dispute     90  0.225                            1.111
fraud-report            50  0.125                            2.000 

message length (words): min 8, median 16, max 24


## 2. Reverse-engineering the generator

The data is synthetic — the starter pack says so — and the raw file shows the generator's
fingerprints. It composes each message from four slots:

```
{greeting} + {body template} + {asset} + {closing}
   "Hey,"     "Can you explain how to move ... to an external wallet?"   "BTC"   "Thanks."
```

Recovering that structure matters because it determines what a train/test split actually
measures. I build **two** groupings and use both later:

- **asset-masked** — only the asset slot collapsed (386 groups, largest 2);
- **template-core** — greeting and closing stripped as well (169 groups, largest 11).

The second is the real template family. A cheap, method-independent cross-check
(connected components of TF-IDF cosine ≥ 0.85) finds a similar picture, which is reassurance
that the regexes are not inventing structure.

In [3]:
from collections import Counter

def norm(t):
    t = re.sub(r"[^a-z ]", " ", t.lower())
    return re.sub(r"\s+", " ", t).strip()

ASSET = (r"\b(btc|eth|sol|ada|doge|dogecoin|bitcoin|ethereum|solana|polygon|matic|xrp|usdc|"
         r"ltc|litecoin|avax|cardano|tron|dot|polkadot)\b")
GREET = r"^(hi|hey|hello team|hello|good morning|quick question|please help|dear support)\b[ ,]*"
CLOSE = (r"\b(thanks|thank you|please advise|appreciate any help|this is time sensitive|"
         r"any help appreciated)\b[ .]*")

normed = [norm(t) for t in texts]
tmpl = [re.sub(ASSET, "ASSET", s) for s in normed]                       # asset slot only
core = [re.sub(r"\s+", " ", re.sub(CLOSE, "", re.sub(GREET, "", s))).strip() for s in tmpl]

for name, keys in [("exact (normalized)", normed), ("asset-masked", tmpl), ("template-core", core)]:
    c = Counter(keys)
    print(f"{name:<20} {len(c):>4} groups | largest {max(c.values()):>2} | "
          f"{sum(v for v in c.values() if v > 1):>3} rows sit in a multi-row group")

biggest = max(Counter(core), key=Counter(core).get)
print(f"\nlargest template family ({Counter(core)[biggest]} rows) - one body, many wrappers:")
for t, k in zip(texts, core):
    if k == biggest:
        print("  -", t)

groups_asset = np.array([{}.setdefault(0, 0) for _ in texts])  # placeholder, replaced below
gid = {}; groups_asset = np.array([gid.setdefault(s, len(gid)) for s in tmpl])
gid = {}; groups = np.array([gid.setdefault(s, len(gid)) for s in core])   # the strict grouping

# sanity: a template family must not span labels, or grouping would be hiding real signal
impure = sum(1 for g in set(groups) if len(set(labels[groups == g])) > 1)
print(f"\ntemplate-core groups: {len(set(groups))} | groups spanning >1 label: {impure} (must be 0)")

exact (normalized)    398 groups | largest  2 |   4 rows sit in a multi-row group
asset-masked          386 groups | largest  2 |  28 rows sit in a multi-row group
template-core         169 groups | largest 11 | 303 rows sit in a multi-row group

largest template family (11 rows) - one body, many wrappers:
  - Hi, Can you explain how to move Ethereum to an external wallet?
  - Please help. Can you explain how to move SOL to an external wallet? This is time sensitive.
  - Hey, Can you explain how to move BTC to an external wallet?
  - Hey, Can you explain how to move ETH to an external wallet? Appreciate any help.
  - Hey, Can you explain how to move Litecoin to an external wallet?
  - Can you explain how to move Dogecoin to an external wallet? Please advise.
  - Quick question, Can you explain how to move ETH to an external wallet? Appreciate any help.
  - Can you explain how to move USDC to an external wallet?
  - Hi, Can you explain how to move XRP to an external wallet? Please advis

In [4]:
# method-independent cross-check: near-duplicate clusters by cosine similarity
from sklearn.metrics.pairwise import cosine_similarity

X_sim = TfidfVectorizer(ngram_range=(1, 2), sublinear_tf=True).fit_transform(normed)
S = cosine_similarity(X_sim)
for thr in [0.95, 0.90, 0.85]:
    n = len(texts); comp = -np.ones(n, int); k = 0
    for i in range(n):
        if comp[i] >= 0:
            continue
        stack, comp[i] = [i], k
        while stack:
            u = stack.pop()
            for v in np.where((S[u] >= thr) & (comp < 0))[0]:
                comp[v] = k; stack.append(v)
        k += 1
    sizes = Counter(comp)
    print(f"cosine >= {thr}: {k} clusters, largest {max(sizes.values())}, "
          f"{sum(v for v in sizes.values() if v > 1)} rows in multi-row clusters")

cosine >= 0.95: 377 clusters, largest 5, 42 rows in multi-row clusters
cosine >= 0.9: 311 clusters, largest 10, 130 rows in multi-row clusters
cosine >= 0.85: 237 clusters, largest 11, 230 rows in multi-row clusters


**Reading:** three-quarters of the corpus sits in a near-duplicate family once the wrapper slots
are stripped — not the 7% the asset-only view suggested. The similarity cross-check agrees.
No family spans two labels, so grouping by family removes leakage without destroying signal.

Both groupings are kept, because they answer different questions (§5).

## 3. Reproducing 98.75% and testing the leakage claim

The shipped script calls `fit_transform` on **all 400 rows and splits afterwards**, so the
TF-IDF vocabulary and IDF weights were computed with the test rows included. That is
preprocessing leakage — the eval saw the test set. Rule: *a valid evaluation must simulate the
information conditions of deployment*; `Pipeline` enforces it.

But I verify the *size* of the effect instead of assuming it.

In [5]:
# (a) exactly as shipped: fit on everything, then split
vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
X_leaky = vec.fit_transform(texts)
Xtr, Xte, ytr, yte = train_test_split(X_leaky, labels, test_size=0.2, random_state=SEED)
leaky = accuracy_score(yte, LogisticRegression(max_iter=2000, C=10.0).fit(Xtr, ytr).predict(Xte))

# (b) same seed, split FIRST, vectorizer fitted on train only
ttr, tte, y2tr, y2te = train_test_split(texts, labels, test_size=0.2, random_state=SEED)
clean = accuracy_score(y2te, baseline_config().fit(ttr, y2tr).predict(tte))

# (c) how different are the learned statistics?
v_all = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(texts)
v_tr  = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(list(ttr))
only_in_full = set(v_all.get_feature_names_out()) - set(v_tr.get_feature_names_out())
idf_all = dict(zip(v_all.get_feature_names_out(), v_all.idf_))
idf_tr  = dict(zip(v_tr.get_feature_names_out(),  v_tr.idf_))

print(f"as shipped (leaky protocol):        {leaky:.4f}")
print(f"split-first, same seed (no leak):   {clean:.4f}")
print(f"vocabulary entries that exist ONLY because the fit saw test rows: {len(only_in_full)}")
print("\nidf drift for sample terms (all-400 fit vs train-only fit):")
for t in ["login", "refund", "fraud"]:
    print(f"  {t:<8} {idf_all[t]:.3f}  vs  {idf_tr[t]:.3f}")

as shipped (leaky protocol):        0.9875
split-first, same seed (no leak):   0.9875
vocabulary entries that exist ONLY because the fit saw test rows: 96

idf drift for sample terms (all-400 fit vs train-only fit):
  login    3.356  vs  3.337
  refund   3.698  vs  3.727
  fraud    4.915  vs  4.980


**Reading:** the protocol is invalid, but removing the leak does not move the score — the IDF
drift is in the third decimal. The honest statement is *"this is a protocol violation whose
effect happens to be negligible on this dataset"* — not "the real number is much lower". The
next sections show why the number survives (the task is at ceiling) and why it still cannot be
trusted as a ship signal.

## 4. What can an 80-row test set certify?

One split, one seed, n=80. Two ways to see the same problem: the spread across seeds, and the
binomial (Wilson) confidence interval around 79/80.

In [6]:
accs = []
for s in range(20):
    a, b, c, d = train_test_split(texts, labels, test_size=0.2, random_state=s)
    accs.append(accuracy_score(d, baseline_config().fit(a, c).predict(b)))
print(f"split-first accuracy across 20 seeds: min {min(accs):.4f}  mean {np.mean(accs):.4f}  max {max(accs):.4f}")

from statsmodels.stats.proportion import proportion_confint
lo, hi = proportion_confint(79, 80, method="wilson")
print(f"Wilson 95% CI for the reported 79/80: [{lo:.3f}, {hi:.3f}]  (width {100*(hi-lo):.1f} pp)")
print("=> '98.75%' is statistically indistinguishable from 94%. One message = 1.25 pp.")

split-first accuracy across 20 seeds: min 0.9625  mean 0.9925  max 1.0000
Wilson 95% CI for the reported 79/80: [0.933, 0.998]  (width 6.5 pp)
=> '98.75%' is statistically indistinguishable from 94%. One message = 1.25 pp.


## 5. Two protocols, two questions — and the gap between them

There is no single "honest" split. Each protocol estimates a different quantity, so I run both
and read the difference:

| Protocol | The question it answers |
|---|---|
| Stratified 5-fold (random) | *If next week's tickets resemble this corpus, how will we do?* |
| Template-grouped 5-fold | *If an entire phrasing family is new to us, how will we do?* |

For a stationary intent like "where are my tax documents", the first is the relevant number.
For **fraud** — adversarial and non-stationary, new scam scripts every month — the second is
the regime that matters. Both use the previous engineer's exact model config.

In [7]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
gkf = GroupKFold(n_splits=5)

pred_strat = cross_val_predict(baseline_config(), texts, labels, cv=skf)
pred_group = cross_val_predict(baseline_config(), texts, labels, cv=gkf, groups=groups)

cmp_rows = []
for name, yp in [("stratified (seen phrasings)", pred_strat), ("template-grouped (unseen)", pred_group)]:
    cmp_rows.append({"protocol": name,
                     "accuracy": accuracy_score(labels, yp),
                     "macro_F1": f1_score(labels, yp, average="macro"),
                     "fraud_recall": recall_score(labels, yp, labels=["fraud-report"], average=None)[0],
                     "errors": int((yp != labels).sum())})
print(pd.DataFrame(cmp_rows).round(4).to_string(index=False))

gap_acc = accuracy_score(labels, pred_strat) - accuracy_score(labels, pred_group)
gap_fr = (recall_score(labels, pred_strat, labels=["fraud-report"], average=None)[0]
          - recall_score(labels, pred_group, labels=["fraud-report"], average=None)[0])
print(f"\nGAP (the memorization component): accuracy {100*gap_acc:.1f} pp | "
      f"fraud recall {100*gap_fr:.1f} pp")

                   protocol  accuracy  macro_F1  fraud_recall  errors
stratified (seen phrasings)    1.0000    1.0000          1.00       0
  template-grouped (unseen)    0.9375    0.9208          0.76      25

GAP (the memorization component): accuracy 6.2 pp | fraud recall 24.0 pp


In [8]:
print("Per-class picture under the STRESS protocol - the view the original report never showed:\n")
print(classification_report(labels, pred_group, digits=3))
print(pd.DataFrame(confusion_matrix(labels, pred_group, labels=ROUTES), index=ROUTES, columns=ROUTES))

from statsmodels.stats.proportion import proportion_confint
k = int(((labels == "fraud-report") & (pred_group == "fraud-report")).sum())
lo, hi = proportion_confint(k, 50, method="wilson")
print(f"\nfraud-report recall (grouped): {k}/50 = {k/50:.3f}, Wilson 95% CI [{lo:.3f}, {hi:.3f}]")

print("\nfraud tickets missed when their whole phrasing family is unseen:")
for t, y, p in zip(texts, labels, pred_group):
    if y == "fraud-report" and p != y:
        print(f"  -> {p:<16} | {t[:86]}")

Per-class picture under the STRESS protocol - the view the original report never showed:

                     precision    recall  f1-score   support

     account-access      0.861     0.990     0.921       100
       fraud-report      1.000     0.760     0.864        50
            general      0.952     1.000     0.976       160
transaction-dispute      0.987     0.867     0.923        90

           accuracy                          0.938       400
          macro avg      0.950     0.904     0.921       400
       weighted avg      0.943     0.938     0.936       400

                     account-access  fraud-report  general  transaction-dispute
account-access                   99             0        0                    1
fraud-report                      6            38        6                    0
general                           0             0      160                    0
transaction-dispute              10             0        2                   78

fraud-report recal

**Reading.** Stratified: perfect. Grouped: accuracy drops **6.2 pp** and **fraud recall falls
from 1.00 to 0.76** — about one fraud ticket in four missed, with a wide interval on 50
examples either way (Wilson [0.63, 0.86]). So the reported 98.75% is *both* unsupported
(invalid protocol) *and* optimistic about genuinely new phrasings.

The misses are diagnostic, not random: phishing reports land in `account-access`, and
"I clicked a fake link and unauthorized transactions are appearing" lands in `general`. §7
shows the mechanism, and §8 tests whether the gap is real or an artifact of the protocol.

## 6. Capacity: 1,522 features for 400 rows

`min_df=1` keeps every term that appears even once. Features that occur in exactly **one**
training document are pure memorization hooks: maximally rare, maximally high-IDF, attachable
to whatever label that one row has. With weak regularization (C=10) the model is free to use
them, and with 3.8 features per row it can draw a perfect boundary — 100% *training* accuracy.

The per-class coefficients show what was actually learned.

In [9]:
vec_full = TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True).fit(texts)
Xf = vec_full.transform(texts)
df_counts = np.asarray((Xf > 0).sum(axis=0)).ravel()
clf_full = LogisticRegression(max_iter=2000, C=10.0).fit(Xf, labels)
feats = np.array(vec_full.get_feature_names_out())

print(f"features: {Xf.shape[1]}  |  rows: {Xf.shape[0]}  |  ratio {Xf.shape[1]/Xf.shape[0]:.1f}:1")
print(f"features appearing in exactly ONE document: {(df_counts == 1).sum()}")
print(f"training accuracy (fit on all 400): {clf_full.score(Xf, labels):.4f}\n")

top = {cls: ", ".join(feats[np.argsort(clf_full.coef_[i])[::-1][:8]])
       for i, cls in enumerate(clf_full.classes_)}
print(pd.Series(top).to_string())

features: 1522  |  rows: 400  |  ratio 3.8:1
features appearing in exactly ONE document: 387
training accuracy (fit on all 400): 1.0000

account-access         login, reset, password, access, code, working,...
fraud-report           someone, account and, was, my account, fraud, ...
general                do, how, what, can, what the, how do, document...
transaction-dispute    but, refund, want, than, shows, but the, the, ...


**Reading:** apart from `fraud` and `refund`, the strongest votes are function words and
template fragments — `someone`, `was`, `never` for fraud; `but`, `shows`, `the` for disputes;
`how do` for general. The model has learned **the fingerprints of a synthetic data generator**,
not the concept of fraud. That is fine for this benchmark and exactly why the score must not be
read as production readiness.

## 7. Adversarial probe: the learned features generate false positives

If Section 6 is right, I should be able to write *benign* messages that trip the classifier by
stuffing them with fraud-vote tokens (`someone`, `my account and`, `never`, `was`) while keeping
the meaning harmless. No search, no gradient tricks — hand-written first attempts:

In [10]:
pipe_full = baseline_config().fit(texts, labels)
probes = [
    "I was wondering if someone on my account and I can both use the app? I never tried before.",
    "My account was set up by someone at your kiosk and I think I never got the welcome email.",
    "Someone told me staking was good. I think my account and wallet never earned anything though?",
    "I think the fraud protection features are great, someone recommended them to me.",
    "Was there ever a time someone could open an account and never verify their email?",
    "I want a refund policy explanation but the app shows nothing than a spinner.",
]
for p in probes:
    pred = pipe_full.predict([p])[0]
    conf = pipe_full.predict_proba([p]).max()
    print(f"pred={pred:<20} conf={conf:.3f} | {p}")

pred=fraud-report         conf=0.574 | I was wondering if someone on my account and I can both use the app? I never tried before.
pred=fraud-report         conf=0.756 | My account was set up by someone at your kiosk and I think I never got the welcome email.
pred=fraud-report         conf=0.948 | Someone told me staking was good. I think my account and wallet never earned anything though?
pred=fraud-report         conf=0.843 | I think the fraud protection features are great, someone recommended them to me.
pred=fraud-report         conf=0.786 | Was there ever a time someone could open an account and never verify their email?
pred=transaction-dispute  conf=0.963 | I want a refund policy explanation but the app shows nothing than a spinner.


**Reading:** five harmless messages classify as `fraud-report` — one at **0.948 confidence**
for what is plainly a staking question — and the `but/shows/refund` probe flips to
`transaction-dispute` at 0.963. First attempts, no optimization.

## 8. The other direction: does it catch *novel* fraud?

§5's grouped protocol says fraud recall is 0.78, but that protocol deletes an entire phrasing
family from training. The deployed model trains on everything, so the fair question is: does it
handle a *new wording of a concept it has seen*? I hand-wrote eight messages the generator
never produced and ran them through the deployed model.

(Caveat stated up front: I wrote these probes, so they may unconsciously favour vocabulary the
model knows, and n=8 is small. Read it as a directional check, not a benchmark.)

In [11]:
novel_fraud = [
    "My balance vanished overnight. Two transfers I never authorised went to an address I don't recognise.",
    "Woke up to 3 outgoing transactions I didn't make. Someone has control of my account, please freeze it.",
    "A caller pretending to be your security team walked me through 'verifying' and now my funds are gone.",
    "Unrecognised device logged in from another country and emptied my wallet while I was asleep.",
    "I've been scammed - gave a code to a fake support agent and my crypto was drained immediately.",
]
novel_other = [
    ("My password reset link expired three times, I still cannot get in.", "account-access"),
    ("The exchange rate on my purchase yesterday was not what the screen quoted, I want it corrected.",
     "transaction-dispute"),
    ("Which networks do you support for stablecoin deposits?", "general"),
]

caught = 0
for t in novel_fraud:
    p = pipe_full.predict([t])[0]; c = pipe_full.predict_proba([t]).max()
    caught += (p == "fraud-report")
    print(f"{'OK  ' if p == 'fraud-report' else 'MISS'} {p:<20} conf={c:.2f} | {t[:66]}")
for t, expected in novel_other:
    p = pipe_full.predict([t])[0]; c = pipe_full.predict_proba([t]).max()
    print(f"{'OK  ' if p == expected else 'MISS'} {p:<20} conf={c:.2f} | {t[:66]}")

print(f"\nnovel fraud caught: {caught}/{len(novel_fraud)}")
print(f"stratified CV predicted fraud recall "
      f"{recall_score(labels, pred_strat, labels=['fraud-report'], average=None)[0]:.2f}; "
      f"grouped CV predicted "
      f"{recall_score(labels, pred_group, labels=['fraud-report'], average=None)[0]:.2f}")

OK   fraud-report         conf=0.52 | My balance vanished overnight. Two transfers I never authorised we
OK   fraud-report         conf=0.70 | Woke up to 3 outgoing transactions I didn't make. Someone has cont
OK   fraud-report         conf=0.79 | A caller pretending to be your security team walked me through 've
OK   fraud-report         conf=0.41 | Unrecognised device logged in from another country and emptied my 
OK   fraud-report         conf=0.71 | I've been scammed - gave a code to a fake support agent and my cry
OK   account-access       conf=0.67 | My password reset link expired three times, I still cannot get in.
OK   transaction-dispute  conf=0.85 | The exchange rate on my purchase yesterday was not what the screen
OK   general              conf=0.97 | Which networks do you support for stablecoin deposits?

novel fraud caught: 5/5
stratified CV predicted fraud recall 1.00; grouped CV predicted 0.76


**Reading — and this is the synthesis of §5, §7 and §8.** All eight novel messages route
correctly, so on *paraphrases of known concepts* the stratified estimate is the better
predictor and the grouped number is pessimistic. Put the three results together and one
mechanism explains all of them:

> **The model is a vocabulary detector.** Novel fraud that *uses* fraud vocabulary is caught
> (§8, 5/5). Benign text that merely *contains* that vocabulary is falsely flagged (§7, 5/6).
> Fraud phrased *without* it — "I clicked a fake link and unauthorized transactions are
> appearing" — is missed (§5).

That is a more useful finding than any single accuracy number, and it sets the real
expectation for production: performance will track how well next month's scam scripts overlap
today's vocabulary — which for an adversarial class is exactly the thing that decays. Hence
fraud recall is reported as a **range, 0.78–1.00**, and the ship decision leans on the review
band (§10) and a shadow pilot rather than on a point estimate.

## 9. Regularization is an operating point

sklearn's `C` multiplies the data term (`C ~ 1/lambda`): **larger C = weaker regularization**.
On a *paraphrased* fraud message that matches no training template, C visibly trades
confidence against prior-collapse:

In [12]:
msg = ["There are trades on my account I never made, what do I do?"]
out = {}
for C in [10.0, 1.0, 0.1]:
    m = make_pipeline(TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True),
                      LogisticRegression(max_iter=2000, C=C)).fit(texts, labels)
    out[f"C={C}"] = pd.Series(m.predict_proba(msg)[0], index=m.classes_).round(3)
print(pd.DataFrame(out))
print("\nC=0.1 under-fits toward the majority class -- and MISSES the fraud.")

                     C=10.0  C=1.0  C=0.1
account-access        0.039  0.121  0.226
fraud-report          0.695  0.429  0.172
general               0.239  0.355  0.406
transaction-dispute   0.027  0.095  0.196

C=0.1 under-fits toward the majority class -- and MISSES the fraud.


## 10. Calibration and the human-review band

The production design I recommend is **selective prediction**: auto-route confident tickets,
send low-confidence ones to a human. That only works if confidence has *dynamic range* — the
model must be measurably less sure when it is wrong. Out-of-fold check:

In [13]:
proba = cross_val_predict(baseline_config(), texts, labels, cv=gkf, groups=groups, method="predict_proba")
conf = proba.max(axis=1)
pred = np.array(sorted(set(labels)))[proba.argmax(axis=1)]
wrong = pred != labels
missed_fraud = (labels == "fraud-report") & wrong

print(f"errors under the stress protocol: {int(wrong.sum())} "
      f"(of which {int(missed_fraud.sum())} are missed fraud)")
print(f"median confidence when RIGHT {np.median(conf[~wrong]):.3f} | "
      f"when WRONG {np.median(conf[wrong]):.3f}\n")
band_rows = []
for thr in [0.90, 0.80, 0.70, 0.60, 0.50]:
    band = conf < thr
    band_rows.append({"threshold": thr,
                      "% tickets to humans": round(band.mean() * 100, 1),
                      "errors caught": f"{int((band & wrong).sum())}/{int(wrong.sum())}",
                      "missed fraud caught": f"{int((band & missed_fraud).sum())}/{int(missed_fraud.sum())}"})
print(pd.DataFrame(band_rows).to_string(index=False))

errors under the stress protocol: 25 (of which 12 are missed fraud)
median confidence when RIGHT 0.876 | when WRONG 0.518

 threshold  % tickets to humans errors caught missed fraud caught
       0.9                 61.5         25/25               12/12
       0.8                 32.2         25/25               12/12
       0.7                 21.8         25/25               12/12
       0.6                 18.8         24/25               11/12
       0.5                  8.5          8/25                4/12


**Reading:** this is the strongest operational result in the notebook. Under the stress
protocol the model is wrong 25 times — and its confidence *knows*: median **0.876 when right
versus 0.518 when wrong**. A `conf < 0.70` band sends **22% of tickets to a human and catches
every single error, including all 12 missed fraud tickets**. (The curve is steep below that:
at 0.50 the band collapses to catching 8 of 25, so the operating point matters.)

So the answer to "fraud recall is only 0.78" is not "pick a better model" — §11 shows no model
choice is provably better — it is **selective prediction**: the system is allowed to say *I am
not sure*, and a human absorbs the residual. That is the design I would ship, and it is why the
`--review-threshold` flag exists on the CLI.

One nuance: `class_weight` distorts probabilities (it deliberately trains on a re-weighted
distribution), so if the review band is the priority, the cleaner stack is *unweighted model +
calibrated probabilities + a threshold chosen from an explicit cost matrix*. Either design
beats silent auto-routing.

## 11. Alternatives — including Naive Bayes — measured, with significance

Naive Bayes is the natural challenger at n=400 (generative models converge faster on small
data). I benchmark it (both feature types), ComplementNB, LinearSVC and gradient boosting under
the *same* stress protocol, then apply **McNemar's exact test** on discordant pairs — the
correct paired test for two classifiers on one dataset. It ignores agreements and asks whether
the disagreements lean one way; with b+c discordant pairs the best achievable two-sided p-value
is 2 x 0.5^(b+c), so significance at 0.05 needs at least 6 disagreements.

(Under the stress protocol there are ~25 errors to work with, so unlike the stratified view
this comparison actually has statistical power.)

In [14]:
T = lambda **kw: TfidfVectorizer(ngram_range=(1, 2), min_df=1, sublinear_tf=True)
models = {
    "LogReg C=10 (shipped config)": baseline_config(),
    "LogReg C=10 + balanced":       baseline_config(class_weight="balanced"),
    "LogReg min_df=2 + balanced":   baseline_config(min_df=2, class_weight="balanced"),
    "MultinomialNB on tf-idf":      make_pipeline(T(), MultinomialNB()),
    "MultinomialNB on counts":      make_pipeline(CountVectorizer(ngram_range=(1, 2)), MultinomialNB()),
    "ComplementNB on tf-idf":       make_pipeline(T(), ComplementNB()),
    "LinearSVC":                    make_pipeline(T(), LinearSVC()),
    "LinearSVC + balanced":         make_pipeline(T(), LinearSVC(class_weight="balanced")),
    "GradientBoosting":             make_pipeline(T(), GradientBoostingClassifier(random_state=SEED)),
}
preds, table = {}, []
for name, m in models.items():
    yp = cross_val_predict(m, texts, labels, cv=gkf, groups=groups)
    preds[name] = yp
    table.append({"model": name,
                  "acc": accuracy_score(labels, yp),
                  "macro_F1": f1_score(labels, yp, average="macro"),
                  "fraud_recall": recall_score(labels, yp, labels=["fraud-report"], average=None)[0],
                  "errors": int((yp != labels).sum())})
print(pd.DataFrame(table).round(4).to_string(index=False))

from scipy.stats import binomtest
base = preds["LogReg C=10 (shipped config)"]
print("\nMcNemar vs shipped config (b = alt fixes an error, c = alt breaks a correct one):")
for name, yp in preds.items():
    if name == "LogReg C=10 (shipped config)":
        continue
    b = int(((base != labels) & (yp == labels)).sum())
    c = int(((base == labels) & (yp != labels)).sum())
    p = binomtest(b, b + c, 0.5).pvalue if b + c else 1.0
    best = binomtest(b + c, b + c, 0.5).pvalue if b + c else 1.0
    print(f"  {name:<28} b={b} c={c:<3} exact p={p:.4f}  (best possible for this b+c: {best:.4f})")

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize
n = NormalIndPower().solve_power(proportion_effectsize(0.96, 0.9375), alpha=0.05, power=0.8,
                                 ratio=1, alternative="larger")
print(f"\nsamples per arm to *prove* 93.75% -> 96.00% at 80% power: {n:,.0f} (we have 400)")

nb_proba = cross_val_predict(models["MultinomialNB on counts"], texts, labels, cv=gkf, groups=groups, method="predict_proba")
print(f"\nNB confidence: {(nb_proba.max(axis=1) > 0.99).mean()*100:.1f}% of predictions above 0.99 "
      f"(LogReg: {(conf > 0.99).mean()*100:.1f}%) -> NB has far less dynamic range for a review band")

                       model    acc  macro_F1  fraud_recall  errors
LogReg C=10 (shipped config) 0.9375    0.9208          0.76      25
      LogReg C=10 + balanced 0.9375    0.9209          0.78      25
  LogReg min_df=2 + balanced 0.9325    0.9144          0.78      27
     MultinomialNB on tf-idf 0.9000    0.8714          0.60      40
     MultinomialNB on counts 0.9125    0.8895          0.74      35
      ComplementNB on tf-idf 0.9225    0.8971          0.70      31
                   LinearSVC 0.9400    0.9233          0.76      24
        LinearSVC + balanced 0.9425    0.9273          0.78      23
            GradientBoosting 0.8800    0.8738          0.76      48

McNemar vs shipped config (b = alt fixes an error, c = alt breaks a correct one):
  LogReg C=10 + balanced       b=3 c=3   exact p=1.0000  (best possible for this b+c: 0.0312)
  LogReg min_df=2 + balanced   b=3 c=5   exact p=0.7266  (best possible for this b+c: 0.0078)
  MultinomialNB on tf-idf      b=0 c=15  exact p=

**Reading — the asymmetry is the point.** Three alternatives are **significantly worse**:
MultinomialNB on TF-IDF (p=0.0001), NB on counts (p=0.013) and gradient boosting (p<0.0001).
The NB failure is instructive: TF-IDF's fractional values break the multinomial count model it
assumes, the likelihood ratios flatten, and the 40% `general` prior swallows the 12.5% fraud
class — imbalance failure in its purest form (ComplementNB exists to correct exactly this and
recovers most of the gap).

But **nothing is significantly better.** Class weighting is b=3 / c=3, p=1.0 — it trades three
errors for three, moving fraud recall 0.76 to 0.78; I keep it because the cost matrix says a
fraud error is worth more than a general one, *not* because the score says so. LinearSVC with
balanced weights posts the best accuracy (0.9425, 23 errors) but at b=3 / c=1, p=0.625 it is
not distinguishable — and it exposes no `predict_proba`, so it cannot drive the review band
from §10 without extra calibration. That is a concrete reason to stay with logistic regression:
**the band is worth more than the 0.5 pp.**

So: **degradations are detectable, improvements are not.** Model choice is not the lever here.
The evaluation instrument, the operating point, and the human-review band are.

## 12. Hygiene candidate: `min_df=2` — tested, adopted

Dropping terms that appear in only one document removes every memorization hook from Section 6.
No improvement is provable (§11), so this must be justified by principle *and shown harmless*:

In [15]:
rows_out = []
for mdf in [1, 2]:
    for cw in [None, "balanced"]:
        m = baseline_config(min_df=mdf, class_weight=cw)
        yp = cross_val_predict(m, texts, labels, cv=gkf, groups=groups)
        nfeat = len(TfidfVectorizer(ngram_range=(1, 2), min_df=mdf, sublinear_tf=True).fit(texts).get_feature_names_out())
        rows_out.append({"min_df": mdf, "class_weight": str(cw), "features": nfeat,
                         "acc": accuracy_score(labels, yp),
                         "macro_F1": f1_score(labels, yp, average="macro"),
                         "fraud_recall": recall_score(labels, yp, labels=["fraud-report"], average=None)[0]})
print(pd.DataFrame(rows_out).round(4).to_string(index=False))
print("\n1,522 -> 1,135 features: all 387 single-document hooks removed for ~0.5 pp accuracy, "
      "with fraud recall unchanged or better. Adopted on principle, not on score.")

 min_df class_weight  features    acc  macro_F1  fraud_recall
      1         None      1522 0.9375    0.9208          0.76
      1     balanced      1522 0.9375    0.9209          0.78
      2         None      1135 0.9325    0.9168          0.76
      2     balanced      1135 0.9325    0.9144          0.78

1,522 -> 1,135 features: all 387 single-document hooks removed for ~0.5 pp accuracy, with fraud recall unchanged or better. Adopted on principle, not on score.


## 13. Embeddings — considered, rejected (for now)

Swapping TF-IDF for pretrained sentence embeddings is easy to wire and impossible to justify
here:

- **Unprovable benefit:** §11 shows no alternative reaches significance in the favourable
  direction, and proving even a ~2 pp gain (93.75% to 96%) at 80% power would need roughly
  1,200 labeled samples per arm. We have 400.
- **Real costs:** a model download and heavier dependency for anyone running the repo, more
  latency per ticket, and a less inspectable failure mode than "which words voted for fraud".
- **The honest counter-argument:** §5/§8 diagnose the model as a *vocabulary detector*, and
  embeddings are exactly the tool for vocabulary mismatch — so this is the most defensible
  upgrade on the table. It is deferred rather than dismissed, with a named trigger:
  measured paraphrase misses on *real* traffic, where the gain would finally be observable.
  (Part B faces the same vocabulary-drift problem in retrieval and solves it structurally,
  with version resolution rather than embeddings — same ladder, same logic.)

## 14. Verdict

**I do not sign off on shipping this as-is.** Five findings, in the order they were measured:

1. **The protocol was invalid.** Vectorizer fitted before the split; one unstratified 80-row
   test set; no error bars. The Wilson 95% CI on 79/80 is [93.3%, 99.8%] — "98.75%" cannot be
   distinguished from 94%.
2. **The number is also optimistic.** The data is generated from ~169 templates
   (greeting + body + asset + closing). Under a random split, siblings of a template sit on
   both sides. Hold whole families out and accuracy falls to **93.75%** with **fraud recall
   0.76** (Wilson 95% CI [0.63, 0.86]) — roughly one fraud ticket in four missed.
3. **The model is a vocabulary detector, not a fraud detector.** One mechanism explains three
   experiments: benign text containing fraud words is flagged (§7, 5 of 6 probes, one at 0.948
   confidence), novel fraud *using* fraud words is caught (§8, 5 of 5), and fraud phrased
   without them — "I clicked a fake link and unauthorized transactions are appearing" — is
   missed (§5). For an **adversarial, drifting** class, that is the property that decays.
4. **The report never examined the class that matters.** No per-class metrics, no confusion
   matrix, and fraud recall on 50 examples carries a ~±12 pp interval either way.
5. **`predict()` retrains per call** and serves a model that was never the one evaluated.

**The numbers I would defend to a PM** — as a range, because the protocol determines the
question:

| | Accuracy | Fraud recall |
|---|---|---|
| Traffic resembling today's mix (stratified) | 100% | 1.00 |
| A genuinely new phrasing family (grouped) | 93.75% | 0.76 |

I would quote **fraud recall 0.76–1.00** and say plainly that where a given month lands depends
on how much next month's scam scripts overlap today's vocabulary. A single point estimate here
would be false precision.

**The production metric:** fraud-report recall with a floor, measured on real traffic — never
on this fixture — at acceptable per-class precision, plus drift monitoring on input text and
confidence distributions, with agent re-routes harvested as free labels.

**The design that makes 0.76 shippable** is not a better model (§11: no alternative is
significantly better; three are significantly worse) but **selective prediction**. Confidence
separates cleanly — median 0.876 when right versus 0.518 when wrong — so a `conf < 0.70` band
routes 22% of tickets to a human and catches **25 of 25 errors, including all 12 missed
frauds** (§10). Ship the model *with* the band, not instead of it.

**The minimal fix** (implemented here, small-diff style): vectorizer inside a `Pipeline`
(leakage structurally impossible), `min_df=2` (removes 387 single-document hooks),
`class_weight='balanced'` (aligns the loss with the cost asymmetry — kept on cost grounds, not
score grounds), evaluation under both protocols with per-class metrics, and `predict()` fits
once and serves many.

**The ship gate:** a shadow pilot on real traffic with the review band enabled. Nothing measured
on a synthetic corpus transfers by default — including the numbers in this notebook.